In [0]:
USE CATALOG olist;

-- ============================================================
-- Query 1: Top 10 product categories by revenue
-- (delivered orders only — filters out cancelled/unavailable)
-- ============================================================
SELECT
  dp.category_name_english,
  ROUND(SUM(f.item_total), 2) AS total_revenue,
  COUNT(*) AS item_count
FROM gold.fact_order_items f
JOIN gold.dim_product dp ON f.product_key = dp.product_key
WHERE f.order_status = 'delivered'
GROUP BY dp.category_name_english
ORDER BY total_revenue DESC
LIMIT 10;


-- ============================================================
-- Query 2: Monthly revenue trend
-- ============================================================
SELECT
  dd.year,
  dd.month,
  ROUND(SUM(f.item_total), 2) AS monthly_revenue
FROM gold.fact_order_items f
JOIN gold.dim_date dd ON f.order_date_key = dd.date_key
WHERE f.order_status = 'delivered'
GROUP BY dd.year, dd.month
ORDER BY dd.year, dd.month;


-- ============================================================
-- Query 3: Outlier detection — orders with abnormally high freight
-- ratio (freight as % of item price), using z-score on freight_ratio
-- ============================================================
WITH freight_ratios AS (
  SELECT
    order_id,
    order_item_id,
    price,
    freight_value,
    CASE WHEN price > 0 THEN freight_value / price ELSE NULL END AS freight_ratio
  FROM gold.fact_order_items
  WHERE order_status = 'delivered' AND price > 0
),
stats AS (
  SELECT AVG(freight_ratio) AS avg_ratio, STDDEV(freight_ratio) AS std_ratio
  FROM freight_ratios
)
SELECT
  fr.order_id,
  fr.order_item_id,
  fr.price,
  fr.freight_value,
  ROUND(fr.freight_ratio, 3) AS freight_ratio,
  ROUND((fr.freight_ratio - s.avg_ratio) / s.std_ratio, 2) AS z_score
FROM freight_ratios fr
CROSS JOIN stats s
WHERE ABS((fr.freight_ratio - s.avg_ratio) / s.std_ratio) > 3
ORDER BY z_score DESC;


-- ============================================================
-- Query 4: Revenue by customer state (geographic breakdown)
-- ============================================================
SELECT
  dc.customer_state,
  ROUND(SUM(f.item_total), 2) AS total_revenue,
  COUNT(DISTINCT f.order_id) AS order_count,
  ROUND(SUM(f.freight_value) / SUM(f.price) * 100, 2) AS freight_pct_of_price
FROM gold.fact_order_items f
JOIN gold.dim_customer dc ON f.customer_key = dc.customer_key
WHERE f.order_status = 'delivered'
GROUP BY dc.customer_state
ORDER BY total_revenue DESC;


-- ============================================================
-- Query 5: Realistic stakeholder ask — "which sellers have the most
-- late-shipped items this quarter, and what's their average item value?"
-- ============================================================
SELECT
  ds.seller_id,
  ds.seller_state,
  COUNT(*) AS item_count,
  ROUND(AVG(f.price), 2) AS avg_item_price
FROM gold.fact_order_items f
JOIN gold.dim_seller ds ON f.seller_key = ds.seller_key
JOIN gold.dim_date dd ON f.order_date_key = dd.date_key
WHERE f.order_status = 'delivered'
  AND dd.year = 2018 AND dd.month IN (7, 8, 9)
GROUP BY ds.seller_id, ds.seller_state
ORDER BY item_count DESC
LIMIT 10;